<a id="from-a-local-notebook-to-a-distributed-rag-system"></a>
# From a Local Notebook to a Distributed RAG System

**AI4Metascience School — CNRS AISSAI Center**
*Domaine Saint-Paul, Saint-Rémy-lès-Chevreuse — Sept/Oct 2026*

*(Continuation of `01_hybrid_RAG_from_scratch_sections1-7.ipynb`)*

---

<a id="where-we-left-off-and-where-were-going"></a>
## Where we left off, and where we're going

In Part 1, everything ran in a single Jupyter kernel, on a single laptop: the vector store, the
embedding model, and the LLM. That's perfect for learning, but it doesn't reflect how a lab would
actually deploy this: heavy components (vector store with a large corpus, LLM inference) belong
on a shared server or your computing center's infrastructure, while end users only need a thin
client.

In this notebook we **split the system across three roles**:

| Role | Runs on | What it does |
|---|---|---|
| **Vector store server** | a remote machine (e.g. a CC front-end node, a lab server, or your workshop partner's laptop) | Hosts ChromaDB in server mode, serves nearest-neighbour queries |
| **LLM server** | a *different* remote machine | Runs Ollama + a LiteLLM proxy in front of it, serves chat completions to authenticated clients |
| **Client** | **your laptop only** | Runs a Streamlit chat UI; embeds queries locally (BGE-M3, lightweight enough); talks to both remote servers over secured connections |

```
                     ┌───────────────────────────┐
                     │   YOUR LAPTOP (client)    │
                     │                           │
                     │  Streamlit UI             │
                     │  BGE-M3 (query encoding)  │
                     │  hybrid_retrieve() logic  │
                     └───────┬───────────┬───────┘
                             │           │
                   SSH tunnel│           │SSH tunnel
                    (encrypted, free)    (encrypted, free)
                             │           │
             ┌───────────────▼────┐   ┌───▼──────────────────┐
             │  MACHINE A         │   │  MACHINE B           │
             │  Chroma server     │   │  Ollama + LiteLLM    │
             │  (dense vectors)   │   │  proxy (chat + auth) │
             │  token-authed      │   │  per-user API keys   │
             └────────────────────┘   └──────────────────────┘
```

Everything here stays **free and open-source**: no paid cloud service, no paid VPN, no paid
license. We use tools you likely already have access to (SSH) plus free open-source software
(ChromaDB server mode, Ollama, LiteLLM proxy).

<a id="practical-setup-for-the-workshop"></a>
## Practical setup for the workshop

You have two ways to do this exercise:

- **Pair exercise (recommended)**: team up with a neighbour. Your partner's laptop becomes
  "Machine A" (vector store) for you, and a third participant's laptop (or the instructor's
  server) becomes "Machine B" (LLM), and vice versa. This is the most realistic and fun version.
- **Solo simulation**: run "Machine A" and "Machine B" as two extra processes on **your own**
  laptop, on different ports, and connect to `127.0.0.1` instead of a real remote IP. All the
  code below is identical either way — only the hostname changes. This is what we'll do by
  default in the instructions so the notebook is self-contained; swap in your partner's SSH
  details to go fully distributed.

<a id="time-budget-for-this-notebook"></a>
## Time budget for this notebook

| Section | Time |
|---|---|
| 8. Networking primer (SSH tunnels) | 15 min |
| 9. Remote vector store (Chroma server + auth) | 25 min |
| 10. Remote LLM (Ollama + LiteLLM proxy, multi-session security) | 30 min |
| 11. Going further: TLS & Tailscale (optional) | 10 min |
| 12. The Streamlit client | 30 min |
| 13. End-to-end test | 10 min |
| **Total** | **~2h00** |

---

## 🎓 About this course — LaboBots School

This notebook continues notebook 1's course, from zero prior RAG/LLM knowledge to a working
Thunderbird mail agent -- whether you administer machines, build data pipelines, or already work
with LLMs day to day. Each section opens with the concept before the code.

**Course objective**: progressively build and deploy the same hybrid RAG pipeline (dense retrieval +
lexical retrieval, fused with Reciprocal Rank Fusion) while exploring several architectures, from
100% local execution to distributed architectures representative of real laboratory environments:
shared GPU resources, institutional computing-center infrastructure, token authentication, and more.

The course is structured around three notebooks, each moving the boundary between what stays in the
laboratory and what is hosted remotely:

| Notebook | Retriever (vector store) | Generation (LLM) | Web interface | Authentication |
|---|---|---|---|---|
| **1. From scratch (local)** | Local — embedded ChromaDB | Local — Ollama | Jupyter notebook (laptop) | None |
| **2. Distributed architecture (GPU)** | Remote — dedicated GPU machine | Remote — same GPU machine (Ollama + LiteLLM proxy, multi-user) | Local — Streamlit (laptop) | Per-participant LiteLLM API key, SSH tunnel |
| **3. Computing-center infrastructure (coming soon)** | Remote — computing-center machine | Remote — LLMs hosted by the computing center | Local — Streamlit (laptop) | Computing-center access token |

This boundary can be adjusted to your laboratory's constraints. For example, you can **keep all
retriever data in the laboratory** (data sovereignty) and use remote computing resources only for
generation, or conversely **host everything remotely** and keep only the web interface in the
laboratory — this is exactly what the course illustrates, notebook by notebook.

**You are here: Notebook 2 — distributed architecture with remote GPU computing.**
The retriever (ChromaDB server) and generation LLM (Ollama behind a LiteLLM proxy for authentication
and multi-user management) are hosted on a remote GPU machine. Only the thin client — Streamlit
interface and local query encoding with BGE-M3 — remains on your laptop, connected to both remote
services through encrypted SSH tunnels.

**Notebook 3 (coming soon)**: the same thin-client approach, but the retriever will be hosted on a
computing-center machine, and generation will use LLMs hosted by the center, accessed with an
authentication token rather than a self-managed Ollama server.

## 🗂️ Plan de ce notebook

- [Where we left off, and where we're going](#where-we-left-off-and-where-were-going)
- [Practical setup for the workshop](#practical-setup-for-the-workshop)
- [Time budget for this notebook](#time-budget-for-this-notebook)
- [8. Networking primer — SSH tunnels](#sec-8)
  - [8.1 Why not expose the ports directly?](#sec-8-1)
  - [8.2 A standard solution: SSH local port forwarding](#sec-8-2)
  - [8.3 Checklist before continuing](#sec-8-3)
- [9. Deploying the vector store on a remote machine](#sec-9)
  - [9.1 On MACHINE A — prepare the database and start Chroma](#sec-9-1)
  - [9.2 On YOUR LAPTOP — connect as a client](#sec-9-2)
  - [9.3 What about the sparse (lexical) index — and the page-level text for small-to-big expansion?](#sec-9-3)
- [10. Deploying the LLM remotely — Ollama + LiteLLM proxy (secured, multi-session)](#sec-10)
  - [10.1 Why not just point Streamlit straight at a remote Ollama?](#sec-10-1)
  - [10.2 LiteLLM Proxy — free & open-source](#sec-10-2)
  - [10.3 🖥️ On MACHINE B (remote) — install and run Ollama + LiteLLM proxy](#sec-10-3)
  - [10.4 🖥️ On MACHINE B — create a virtual API key per participant](#sec-10-4)
  - [10.5 💻 On YOUR LAPTOP — call the remote LLM through the proxy](#sec-10-5)
  - [10.6 Checking your usage / budget](#sec-10-6)
- [11. Going further (optional): TLS certificates & Tailscale](#sec-11)
  - [11.1 Caddy — free, automatic HTTPS reverse proxy](#sec-11-1)
  - [11.2 Tailscale — free tier, zero-config mesh VPN](#sec-11-2)
- [12. Building the Streamlit client](#sec-12)
  - [12.1 Why Streamlit, and what it will (and won't) do](#sec-12-1)
  - [12.2 Writing the app file](#sec-12-2)
  - [12.3 Configuration: a secrets.toml file instead of hardcoded keys](#sec-12-3)
  - [12.4 💻 Launch the Streamlit app](#sec-12-4)
- [13. End-to-end checklist](#sec-13)
- [14. Bonus: a secured, multi-user, themed client](#sec-14)
  - [14.1 Per-user LiteLLM key vault](#sec-14-1)
  - [14.2 A theme, instead of Streamlit's defaults](#sec-14-2)
  - [14.3 The app: auth gate, per-user key, themed sidebar](#sec-14-3)
  - [14.4 Launch, and what to check](#sec-14-4)
- [What's next](#whats-next)


<p align="center">
  <img src="assets/diagrams/distributed_architecture.svg" alt="Architecture distribuee du workshop" width="720"><br>
  <sub><i>Vue d'ensemble : le laptop du participant (Streamlit) parle a ChromaDB et LiteLLM via un tunnel SSH ; la machine distante heberge ChromaDB, Ollama, LiteLLM et Postgres</i></sub>
</p>


<a id="sec-8"></a>
## 8. Networking primer — SSH tunnels

<a id="sec-8-1"></a>
### 8.1 Why not expose the ports directly?

ChromaDB and the LiteLLM proxy are HTTP services bound to remote loopback interfaces. We keep them
there and access them through an encrypted SSH connection. This avoids exposing unauthenticated API
ports directly to the network.

<a id="sec-8-2"></a>
### 8.2 The workshop topology

The whole practical lab uses one laptop and one remote SSH host:

```text
YOUR LAPTOP                                  REMOTE SSH HOST
127.0.0.1:8000  -- SSH local forward -->     127.0.0.1:8000 ChromaDB
127.0.0.1:4000  -- SSH local forward -->     127.0.0.1:4000 LiteLLM proxy

SSH: labobots@195.221.220.18:22003
Slurm partition label: labobots
```

The client code always connects to `127.0.0.1`. The SSH tunnel carries that traffic to the remote
loopback services. The automated setup in Section 8.4 creates both forwards.

For reference, the two manual commands are:

```bash
ssh -p 22003 -N \
  -o ExitOnForwardFailure=yes \
  -o ServerAliveInterval=30 \
  -L 127.0.0.1:8000:127.0.0.1:8000 \
  labobots@195.221.220.18
```

```bash
ssh -p 22003 -N \
  -o ExitOnForwardFailure=yes \
  -o ServerAliveInterval=30 \
  -L 127.0.0.1:4000:127.0.0.1:4000 \
  labobots@195.221.220.18
```

Keep the two sessions running, or execute the single setup cell in Section 8.4. SSH authentication
protects the tunnel; LiteLLM still requires its own participant API key.

<a id="sec-8-3"></a>
### 8.3 Checklist before continuing

- [ ] You can SSH to `labobots@195.221.220.18` on port `22003`.
- [ ] The remote `labobots` partition is visible with `sinfo`, when Slurm is exposed on the host.
- [ ] The remote Chroma service is bound to `127.0.0.1:8000`.
- [ ] The remote LiteLLM proxy is bound to `127.0.0.1:4000`.
- [ ] Local ports `8000` and `4000` are available.
- [ ] Your notebook kernel and SSH tunnels run on the same laptop.
- [ ] You have a LiteLLM participant key.

<a id="sec-8-4"></a>
### 8.4 One-command setup from your laptop

Use the executable setup cell below. It copies the local Chroma database only when the remote copy is
absent, prepares Chroma, starts the remote service, and opens both local forwards.

### 8.4 One-command setup from your laptop

There are two different operations and they must not be run by all participants:

- **Instructor/server preparation, once:** from the local workspace terminal, run
  `./rag_workshop/manage_remote_rag.sh prepare`. The script discovers the authenticated remote
  user's real home directory, copies Chroma only if absent, and starts the shared service.
- **Participant setup, once per laptop:** run
  `./rag_workshop/manage_remote_rag.sh tunnel` from the local workspace. Participants never need
  to open a shell on the server or know its filesystem paths.

The script uses the SSH account `labobots@195.221.220.18` on port `22003`. It stores the database below
the remote account's actual `$HOME`, not an assumed `/home/labobots` path.

If an old failed operation left a lock, the administrator must first confirm that no copy is active,
then run:

```bash
./rag_workshop/manage_remote_rag.sh unlock
```

For a local port conflict, choose alternative local ports without changing the remote services:

```bash
./rag_workshop/manage_remote_rag.sh tunnel --chroma-port 18000 --litellm-port 14000
```

> Run `prepare` only once, preferably before the class. All participants use their own local terminal
> and only run `tunnel`; the remote server remains transparent to them.

In [1]:
%%bash
# Administrator-only preparation, run from the local workspace root.
# The script discovers the authenticated user's actual remote $HOME.
./rag_workshop/manage_remote_rag.sh prepare

Remote workspace: /mnt/backup/labobots/rag_workshop
Remote database already exists; nothing copied. Use 'copy --force' to replace it.
Remote Chroma environment already exists.
Chroma is already running on remote port 8000.


### 8.5 Scaling to 32 participants

With the combined tunnel above, each participant opens one authenticated SSH connection carrying two
forwards. The local ports are private to each laptop, so all participants may use `8000` and `4000`
without conflicts. They do **not** share a local port namespace.

The server still needs capacity for:

- at least 32 simultaneous authenticated SSH connections, plus administrative sessions;
- an `sshd` policy whose `MaxStartups` and connection limits tolerate the class startup burst;
- enough file descriptors, CPU, memory, and network bandwidth;
- LiteLLM/Ollama concurrency and queue limits appropriate for 32 requests;
- one participant API key per user, with budgets and rate limits configured in LiteLLM.

Ask the administrator to check `MaxStartups`, `MaxSessions`, `ulimit -n`, and any firewall or gateway
connection limits. Have participants start the setup cell in small waves rather than all pressing Run
at exactly the same second. A failure to open the SSH tunnel is not fixed by changing the local ports;
it usually means an SSH server admission limit, authentication problem, or an occupied local port.

> The `labobots` Slurm partition is a scheduling/resource label, not a tunnel fan-out mechanism. If
> services run on a compute node instead of `195.221.220.18`, the administrator must provide a
> site-specific `ProxyJump`, node allocation, or gateway endpoint before this tunnel cell can work.

### 8.6 Reproducible environments and emergency operator script

The workspace contains `pyproject.toml`, `uv.lock`, `rag_workshop/setup_uv.sh`, and
`rag_workshop/manage_remote_rag.sh`. Together they define the laptop environment and the remote
Chroma recovery workflow.

Prepare the laptop once from the workspace root:

```bash
./rag_workshop/setup_uv.sh
```

Select the kernel `Python (LaboBots RAG workshop)` in VS Code and run project commands with `uv run`.
The administrator and participants still run every command from the laptop. The operator script uses
`uv` for the remote Chroma environment when `uv` is available on the server; its fallback is a pinned
virtual environment, so a server without uv remains supported.

The operator tool supports safe operational actions:

```text
 doctor       check local tools, SSH access, database, and remote status
 prepare      copy the database only if absent, then start Chroma
 copy         copy only if absent
 copy --force backup the remote database, replace it, and keep the backup
 start/stop   start or stop the shared remote Chroma process
 restart      stop and start Chroma
 status       inspect the local tunnel and remote Chroma
 verify       compare local and remote Chroma
 tunnel       open both forwards in one SSH connection
 stop-tunnel stop the local tunnel
```

Run `prepare` only once as administrator; each participant normally runs only `tunnel`. If local
ports `8000` or `4000` are blocked, choose alternatives without changing remote ports:

```bash
./rag_workshop/manage_remote_rag.sh tunnel --chroma-port 18000 --litellm-port 14000
```

In [2]:
%%bash
# Safe local check: display the operator commands without connecting to the server.
./rag_workshop/manage_remote_rag.sh --help

Usage:
  manage_remote_rag.sh doctor
  manage_remote_rag.sh prepare
  manage_remote_rag.sh copy [--force]
  manage_remote_rag.sh start
  manage_remote_rag.sh stop
  manage_remote_rag.sh restart
  manage_remote_rag.sh status
  manage_remote_rag.sh verify [--all-ids]
  manage_remote_rag.sh tunnel [--chroma-port PORT] [--litellm-port PORT]
  manage_remote_rag.sh stop-tunnel
  manage_remote_rag.sh unlock

Environment overrides:
  REMOTE_USER, REMOTE_HOST, REMOTE_PORT, REMOTE_ROOT, REMOTE_VENV
  REMOTE_CHROMA_PORT, REMOTE_LITELLM_PORT
  LOCAL_CHROMA_PORT, LOCAL_LITELLM_PORT, LOCAL_DB

Examples:
  ./rag_workshop/manage_remote_rag.sh prepare
  ./rag_workshop/manage_remote_rag.sh copy --force
  ./rag_workshop/manage_remote_rag.sh tunnel --chroma-port 18000 --litellm-port 14000
  ./rag_workshop/manage_remote_rag.sh stop-tunnel


<p align="center">
  <img src="https://media.geeksforgeeks.org/wp-content/uploads/20191031165032/ssh_local_port3.jpg" alt="SSH local port forwarding diagram" width="480"><br>
  <sub><i>SSH local port forwarding — the mechanism used to reach Machine A and Machine B — [GeeksforGeeks](https://www.geeksforgeeks.org/computer-networks/ssh-port-forwarding/)</i></sub>
</p>

<a id="sec-9"></a>
## 9. Deploying the vector store on the remote workshop host

<a id="sec-9-1"></a>
### 9.1 Remote preparation: `labobots@195.221.220.18`

The practical setup uses the single SSH host below for the remote services:

```bash
ssh -p 22003 labobots@195.221.220.18
```

The executable setup cell in Section 8.4 performs the database copy and starts Chroma automatically.
It uses the local Chroma version when it can detect one, which reduces the risk of opening a
persisted database with an incompatible client/server version.

The database is stored remotely under:

```text
$HOME/rag_workshop/chroma_db
```

Chroma listens only on the remote loopback interface:

```bash
chroma run --host 127.0.0.1 --port 8000 --path "$HOME/rag_workshop/chroma_db"
```

The `labobots` partition is checked by the setup cell when Slurm's `sinfo` command is available. The
workshop default deliberately starts the service on the SSH host itself, because a Slurm compute node
requires a site-specific node allocation and a corresponding `ProxyJump` or forwarding route.

> **Do not merge databases.** If `$HOME/rag_workshop/chroma_db/chroma.sqlite3` already exists, the
> setup cell does not overwrite it. Remove or rename it remotely only after confirming that replacing
> the data is intended.

<a id="sec-9-2"></a>
### 9.2 On your laptop: connect through the local tunnel

After Section 8.4, Chroma is reachable locally at `http://127.0.0.1:8000`:

```python
import chromadb

client = chromadb.HttpClient(host="127.0.0.1", port=8000, ssl=False)
print("Server heartbeat:", client.heartbeat())
print("Available collections:", client.list_collections())
```

The heartbeat confirms that the local forward reaches the remote Chroma process. If the collection
is missing, inspect the remote `chroma.log` and verify that the database directory was copied before
Chroma started.

<p align="center">
  <img src="https://github.com/chroma-core/chroma/raw/main/docs/assets/chroma-wordmark-color.png" alt="ChromaDB logo" width="260"><br>
  <sub><i>ChromaDB runs here in server mode on Machine A</i></sub>
</p>

In [3]:
import chromadb

remote_chroma_client = chromadb.HttpClient(
    host="127.0.0.1",
    port=8000,
    ssl=False,
)

print("Heartbeat :", remote_chroma_client.heartbeat())

remote_collection = remote_chroma_client.get_collection(
    name="ccin2p3_docs",
    embedding_function=None,
)

print(f"Collection : {remote_collection.name}")
print(f"Nombre de chunks : {remote_collection.count()}")

Heartbeat : 1790140238710591640
Collection : ccin2p3_docs
Nombre de chunks : 813


In [4]:
# Run this on YOUR LAPTOP after Section 8.4 has created the local tunnel.
# The local endpoint 127.0.0.1:8000 forwards to Chroma on labobots@195.221.220.18:22003.

import chromadb

CHROMA_HOST = "127.0.0.1"
CHROMA_PORT = 8000

# No token or Chroma auth provider is configured; SSH protects the loopback-only service.
remote_chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)

try:
    collections = remote_chroma_client.list_collections()
    print("Connected to remote Chroma server. Collections found:", [c.name for c in collections])
except Exception as e:
    print("Could not reach the remote Chroma server.")
    print("Checklist: run Section 8.4, and check ~/rag_workshop/chroma.log on the server.")
    print("Error:", e)

Connected to remote Chroma server. Collections found: ['ccin2p3_docs']


In [5]:
import chromadb
from FlagEmbedding import BGEM3FlagModel

# Connect through the SSH tunnel from Section 8.
remote_chroma_client = chromadb.HttpClient(
    host="127.0.0.1",
    port=8000,
    ssl=False,
)

# Fetch the existing collection from Part 1.
COLLECTION_NAME = "ccin2p3_docs"
remote_collection = remote_chroma_client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=None,  # We supply embeddings ourselves.
)

chunk_count = remote_collection.count()
print(f"Remote collection '{COLLECTION_NAME}' has {chunk_count} chunks.")

if chunk_count == 0:
    raise ValueError(
        "The remote collection is empty. Check the copied database "
        "directory and the server's --path setting."
    )

# Encode the query locally using the same model as in Part 1.
bge_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)

test_query = "How do I submit a job with SLURM?"

encoded_query = bge_model.encode(
    [test_query],
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)
test_query_emb = encoded_query["dense_vecs"][0]

# Search the remote vector store.
result = remote_collection.query(
    query_embeddings=[test_query_emb.tolist()],
    n_results=min(3, chunk_count),
    include=["documents", "metadatas", "distances"],
)

# Display the results.
for rank, (doc, meta, dist) in enumerate(
    zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ),
    start=1,
):
    source = (meta or {}).get("source_title") or "Unknown source"
    distance = f"{dist:.4f}" if dist is not None else "n/a"

    print(f"#{rank}  distance={distance}  source={source}")
    if doc:
        print(doc[:300].replace("\n", " "))
    print()

/home/anne/Nextcloud/LaboBots_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Remote collection 'ccin2p3_docs' has 813 chunks.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15268.06it/s]


#1  distance=0.3736  source=Soumettre un job — CC-IN2P3
Les limites supérieurs de ces paramètres seront discutés dans le paragraphe Limites des paramètres obligatoires . -t -c --mem= Contrôlez que votre demande de ressources ne dépasse pas les limites matérielles des nœuds de la plateforme de calcul. Contrôlez que votre demande de ressources ne dépasse p

#2  distance=0.3750  source=Soumettre un job — CC-IN2P3
À la soumission, des informations vous sont retournées dans la sortie standard : le groupe de calcul ( Account , ici ccin2p3 ), le nœud de soumission (ici cca013 ), la partition htc (par défaut) dans laquelle le job sera exécuté, ou encore l’identifiant du job contenu dans la variable d’environnemen

#3  distance=0.3910  source=Suivi des jobs — CC-IN2P3
Veuillez trouver dans le projet Gitlab SLURM scripts un exemple pour lancer le job de profilage Note Il est possible de profiler un job interactif : Lancez la session ajoutant l’option --profile=task dans votre ligne de commande :

<a id="sec-9-3"></a>
### 9.3 Local sparse index and page-level context

For this workshop, the laptop keeps the sparse/lexical index and page-level text used for small-to-big
context expansion. Chroma's dense collection is remote; the local files remain available to the hybrid
retrieval code:

```text
rag_workshop/corpus/lexical_weights.pkl
rag_workshop/corpus/chunks.pkl
rag_workshop/corpus/page_full_text_by_url.pkl
```

The setup cell copies only `rag_workshop/chroma_db` to the remote host. These three corpus artifacts
are already local and do not need a second transfer. The sparse scoring, RRF fusion, and context
construction code from Notebook 1 stays unchanged; only dense search targets `remote_collection`.

<a id="sec-10"></a>
## 10. Deploying the LLM on the remote workshop host

<a id="sec-10-1"></a>
### 10.1 Why not point the laptop directly at Ollama?

*(New to what an LLM is, or what Ollama actually does? See notebook 1, Section 1.0 -- a 60-second
primer before this section assumes you already know.)*

Ollama's HTTP API has no built-in participant authentication, per-user budgets, or usage tracking.
The remote host therefore runs Ollama behind a LiteLLM proxy. The laptop reaches only the proxy
through the SSH forward on local port `4000`; Ollama remains on remote loopback port `11434`.

<a id="sec-10-2"></a>
### 10.2 LiteLLM proxy — free and open-source

LiteLLM exposes an OpenAI-compatible `/v1/chat/completions` endpoint with virtual API keys, budgets,
rate limits, and request logging. [LiteLLM's own architecture diagram](https://docs.litellm.ai/docs/)
(proxy docs, top of page) shows this same "one proxy in front of many possible backends" shape --
Ollama is just one of the many LLM backends it can sit in front of. The model backend and proxy both
run on the same remote host here:

```text
Laptop 127.0.0.1:4000  <-- SSH -->  Remote 127.0.0.1:4000 LiteLLM
                                      Remote 127.0.0.1:11434 Ollama
SSH: labobots@195.221.220.18:22003
Slurm partition label: labobots
```

**What the proxy buys you, visually** -- every participant gets their own key, but they all
share the one loaded Ollama model on Machine B, instead of each running their own copy:

```mermaid
flowchart LR
    subgraph Participants
        C1["Participant 1\n(key sk-...01)"]
        C2["Participant 2\n(key sk-...02)"]
        C3["Participant N\n(key sk-...NN)"]
    end
    C1 --> LL["LiteLLM proxy\n(per-key auth, budget,\nrate limit, logging)"]
    C2 --> LL
    C3 --> LL
    LL -- "one shared connection" --> O["Ollama\n(llama3.2:3b)"]
```

<a id="sec-10-3"></a>
### 10.3 On the remote host — install and run Ollama + LiteLLM

Connect to the remote host with:

```bash
ssh -p 22003 labobots@195.221.220.18
```

The instructor or server administrator prepares PostgreSQL, Ollama, the model, and LiteLLM on the
remote host -- in that order, since LiteLLM's virtual-key management (Section 10.4) needs a
Postgres-backed database to persist keys, budgets, and spend logs. PostgreSQL itself is installed
by `manage_litellm.sh install-db` (Section 10.6, needs sudo once); what follows is what that
command and `manage_litellm.sh start` actually do on the remote host:

```bash
# Ollama and its model are installed once by the server administrator.
ollama pull llama3.2:3b

# Same tool as the laptop (Section 0 of notebook 00) -- a plain venv here, not the project one.
curl -LsSf https://astral.sh/uv/install.sh | sh   # if uv isn't already on the remote host
uv venv "$HOME/litellm-venv"
uv pip install --python "$HOME/litellm-venv/bin/python" "litellm[proxy]"
```

Create `$HOME/rag_workshop/litellm_config.yaml`, with `database_url` pointing at the `litellm`
role and database that `install-db` created (the password is generated by `install-db` and printed
once -- never hardcode it here):

```yaml
model_list:
  - model_name: workshop-llm
    litellm_params:
      model: ollama/llama3.2:3b
      api_base: http://127.0.0.1:11434

general_settings:
  master_key: "sk-admin-change-me"
  database_url: "postgresql://labobots:<password from install-db>@localhost:5432/litellm"
```

Start Ollama and the proxy according to the server's service policy. Always pass `--host
127.0.0.1`: every example in this notebook assumes the proxy is reachable only through the SSH
tunnel (Section 8.4/9) at `127.0.0.1:4000`, but LiteLLM's own CLI default is `--host 0.0.0.0` --
listening on every network interface -- which is not what this workshop's tunnel-only design
intends:

```bash
"$HOME/litellm-venv/bin/litellm" --config "$HOME/rag_workshop/litellm_config.yaml" \
    --port 4000 --host 127.0.0.1
```

The setup cell in Section 8.4 opens the tunnel to this remote proxy. It does not copy or print API
keys and does not guess a site-specific Slurm allocation.

> **In practice**, you will not type any of the commands above by hand: `manage_litellm.sh
> install-db` then `manage_litellm.sh start` (Section 10.6) run all of it for you -- generating
> and storing the Postgres password, writing the config with `--host 127.0.0.1` already set, and
> starting LiteLLM with `uv`. This section exists so you know what those two commands are doing
> on the remote host, not as a second, separate setup path to actually run.

<a id="sec-10-4"></a>
### 10.4 Create a virtual API key per participant

With the proxy running on the remote host, issue one key per participant through the remote loopback
API:

```bash
curl -X POST http://127.0.0.1:4000/key/generate \
  -H "Authorization: Bearer sk-admin-change-me" \
  -H "Content-Type: application/json" \
  -d '{
        "duration": "8h",
        "max_budget": 5,
        "models": ["workshop-llm"],
        "key_alias": "participant-01"
      }'
```

Keep the returned participant key private. SSH authentication grants tunnel access; it does not
replace the LiteLLM API key.

<a id="sec-10-5"></a>
### 10.5 On your laptop — call the remote LLM through the tunnel

After Section 8.4, the remote proxy is reachable locally at `http://127.0.0.1:4000`. The client uses
an environment variable or a hidden prompt for the participant key; no key is hardcoded in source.

> There is no OpenAI account or OpenAI API key in this workshop. `requests` talks to the local port
> forwarded to your own LiteLLM proxy.

### 10.6 Administrator: start LiteLLM and generate participant keys

The port-4000 error means LiteLLM is not running yet. The administrator can prepare it and generate all
36 participant keys from the **laptop**, without opening a remote shell manually:

```bash
./rag_workshop/manage_litellm.sh status
./rag_workshop/manage_litellm.sh start
./rag_workshop/manage_litellm.sh create-keys --count 36
```

The script asks for the LiteLLM master key without displaying it. It stores the generated keys remotely
in `$HOME/rag_workshop/participant-keys.tsv` with permissions `600`; distribute each row privately.
Participants use their own key through `LITELLM_KEY` or the notebook prompt.

Before starting, the administrator must ensure that Ollama is installed and `llama3.2:3b` is available
on the server. Also check storage first: the server reported `/mnt/backup` at approximately 94% usage,
so large model or package downloads may fail.

> Never commit `participant-keys.tsv`, `litellm_config.yaml`, `litellm.db`, or any master/participant key.

<p align="center">
  <img src="https://raw.githubusercontent.com/ollama/ollama/main/docs/ollama-logo.svg" alt="Ollama logo" width="70"><br>
  <sub><i>Ollama — serves the model on Machine B</i></sub>
</p>

<p align="center">
  <img src="https://docs.litellm.ai/img/logo.svg" alt="LiteLLM logo" width="220"><br>
  <sub><i>LiteLLM proxy — key authentication, budgets, multi-user access</i></sub>
</p>

> The cell below will prompt you for **your LiteLLM key** (from `participant-keys.tsv`) -- this is
> a different credential from the Streamlit login you may be issued later in Section 14.3, if the
> instructor is running the secured, multi-user client. Two separate prompts for two separate
> things is expected, not a bug: this key lets you call the shared LLM proxy directly from this
> notebook; the Section 14 login is for the web app itself.

In [6]:
# Run on your laptop, with the SSH tunnel to the server active.

import os
from getpass import getpass

import requests

LITELLM_PROXY_URL = "http://127.0.0.1:4000"
LITELLM_MODEL_NAME = "workshop-llm"


def call_remote_llm(
    messages: list,
    model: str = LITELLM_MODEL_NAME,
) -> str:
    """Call the LiteLLM proxy through the SSH tunnel."""
    response = requests.post(
        f"{LITELLM_PROXY_URL}/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {MY_PARTICIPANT_KEY}",
        },
        json={
            "model": model,
            "messages": messages,
        },
        timeout=(5, 120),
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


# Read the participant key from the environment; if missing OR rejected by the proxy, prompt for
# it again instead of crashing outright -- a typo is far more likely than "there's truly no way
# to proceed". `getpass` renders a masked inline input field in the notebook (Jupyter/VS Code),
# not a separate OS window, without ever displaying the key in the cell's own output.
# MY_PARTICIPANT_KEY is read by call_remote_llm above as a plain global, same as before -- only
# the retry loop around it is new.
MY_PARTICIPANT_KEY = os.environ.get("LITELLM_KEY", "").strip()
MAX_KEY_ATTEMPTS = 3

for attempt in range(1, MAX_KEY_ATTEMPTS + 1):
    if not MY_PARTICIPANT_KEY:
        MY_PARTICIPANT_KEY = getpass("Your LiteLLM participant key: ").strip()
    if not MY_PARTICIPANT_KEY:
        print(f"No key entered ({attempt}/{MAX_KEY_ATTEMPTS}).")
        continue
    try:
        answer = call_remote_llm([
            {
                "role": "system",
                "content": "You are a concise assistant.",
            },
            {
                "role": "user",
                "content": "In one sentence, what is Reciprocal Rank Fusion?",
            },
        ])
        break
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code in (401, 403):
            print(
                f"That key was rejected by the proxy ({e.response.status_code}) -- check for a "
                f"typo, or ask the instructor if it might be expired/revoked "
                f"({attempt}/{MAX_KEY_ATTEMPTS})."
            )
            MY_PARTICIPANT_KEY = ""  # clear it so the next loop iteration prompts again
            continue
        raise  # any other HTTP error (network, proxy down, 500, ...) is a real problem, not a bad key
else:
    raise ValueError(
        f"No valid LiteLLM participant key after {MAX_KEY_ATTEMPTS} attempts. Re-run this cell "
        "to try again, or double-check your key in participant-keys.tsv / with the instructor."
    )

print(answer)

Reciprocal Rank Fusion (RRF) is a ranking fusion method that combines the ranks of two or more ranked lists to produce a single, optimal ranked list by considering the reciprocal rank relationships between corresponding items in the input lists.


<a id="sec-10-6"></a>
### 10.6 Checking your usage / budget

Since LiteLLM tracks usage per key, you (or the instructor) can check remaining budget at any
time — useful to understand what "secured multi-session" buys you in practice:

```bash
curl -X GET "http://127.0.0.1:4000/key/info?key=sk-your-key" \
  -H "Authorization: Bearer sk-admin-change-me"
```

<a id="sec-10-7"></a>
### 10.7 What gets logged server-side (and what to tell participants)

Two different things are called "history" in this workshop, and they are easy to conflate:

1. **Client-side conversation memory** (Section 12/14): the Streamlit app resends the last
   `MAX_HISTORY_TURNS` question/answer pairs to the LLM so follow-up questions work. This lives
   only in that browser tab's `st.session_state` -- gone on refresh, never written to disk.
2. **Server-side request logging**: LiteLLM records **every** request it proxies -- full
   `messages` (the question, with retrieved context) and `response` (the answer), plus the
   calling `api_key`, token counts, and cost -- into the `LiteLLM_SpendLogs` table in the
   Postgres database set up in Section 10.4. This happens **by default**, for every key, unless
   `litellm_settings.turn_off_message_logging: true` is set in `litellm_config.yaml` -- which it
   is not, in this workshop's config.

That second point is worth being deliberate about:

- It's genuinely useful: as the instructor, you can review afterwards what participants asked,
  which questions the corpus failed to answer well, etc. -- real pedagogical signal.
- But it means every participant's verbatim questions (and the assistant's answers) are stored
  centrally, tied to their personal key. **Tell participants this before the workshop** --
  e.g. a one-line notice alongside where you hand out keys (Section 10.4): *"Questions you ask
  the assistant are logged server-side for pedagogical review after the session; avoid entering
  personal or sensitive information."* Treat this the same way you'd treat any lab tool that
  logs usage -- a matter of disclosure, not of hiding it.

If you'd rather not keep this data at all, disable it instead: add `turn_off_message_logging:
true` under `litellm_settings` in `manage_litellm.sh`'s generated config, restart LiteLLM, and
only spend/token counts (no content) will be recorded going forward.

**Reviewing logs after the workshop** (run on the remote host, as the `labobots` account):

```bash
psql -h 127.0.0.1 -U labobots -d litellm -c "
  SELECT api_key, model, messages, response, \"startTime\"
  FROM \"LiteLLM_SpendLogs\"
  ORDER BY \"startTime\" DESC
  LIMIT 20;
"
```

`api_key` here is LiteLLM's own (hashed/masked) token, not the raw `sk-...` value -- cross-
reference it with `key_alias` via `/key/info` (Section 10.6) if you need to map a row back to a
specific `participant-NN` alias from `participant-keys.tsv`.

<a id="sec-11"></a>
## 11. Going further (optional): TLS certificates & Tailscale

SSH tunnels are perfect for this workshop, but they require an active tunnel per session, opened
manually by someone with SSH access. Two free alternatives worth knowing about if you deploy this
for your lab longer-term:

<a id="sec-11-1"></a>
### 11.1 [Caddy](https://caddyserver.com/) — free, automatic HTTPS reverse proxy

Caddy is a free, open-source web server that gets you valid TLS certificates **automatically**
(via Let's Encrypt) with a two-line config, and can sit in front of both the Chroma server and
the LiteLLM proxy:

```
# Caddyfile on Machine A
vectorstore.yourlab.example.org {
    reverse_proxy localhost:8000
}
```

This requires a real DNS name pointing at Machine A and an open port 443 — appropriate for a
proper internal service, less so for an ad hoc workshop pairing exercise.

<a id="sec-11-2"></a>
### 11.2 [Tailscale](https://tailscale.com/) — free tier, zero-config mesh VPN

Tailscale creates a private, encrypted (WireGuard) network between your devices, each getting a
stable private IP/hostname, **without** opening any ports on your router or needing a public DNS
entry. The free tier covers up to 100 devices/3 users, which is more than enough for a lab or a
workshop. Once installed and logged in on both machines and your laptop:

```bash
# On Machine A / Machine B / your laptop, after `tailscale up`:
tailscale status   # shows the private hostnames, e.g. machine-a.tailnet-name.ts.net
```

You would then connect directly to `machine-a.tailnet-name.ts.net:8000` — no SSH tunnel needed,
still fully encrypted, still free. This is a very good default recommendation for a lab that
wants this RAG system to be reachable by multiple people over time without manually managing SSH
tunnels.

> For the rest of *this* notebook, we stick with the SSH tunnel approach from Section 8 — it
> needs nothing installed beyond what you already have, which matters for a one-day workshop.

<p align="center">
  <img src="https://raw.githubusercontent.com/ollama/ollama/main/docs/ollama-logo.svg" alt="Ollama logo" width="70"><br>
  <sub><i>Ollama — serves the model on Machine B</i></sub>
</p>

<p align="center">
  <img src="https://docs.litellm.ai/img/logo.svg" alt="LiteLLM logo" width="220"><br>
  <sub><i>LiteLLM proxy — key authentication, budgets, multi-user access</i></sub>
</p>

<a id="sec-12-3"></a>
### 12.3 Configuration: a `secrets.toml` file instead of hardcoded keys

Rather than hardcoding `MY_PARTICIPANT_KEY` (or anything else personal) directly in
`streamlit_app.py`, we use Streamlit's built-in
[`secrets.toml`](https://docs.streamlit.io/develop/concepts/connections/secrets-management)
mechanism: a small config file, kept **outside** the Python source, that Streamlit loads
automatically into `st.secrets`. This means:

- You edit **one short TOML file**, not the generated Python app.
- It's easy to `.gitignore` (Streamlit's own project template already does this), so a personal
  key never accidentally ends up committed anywhere.
- The Chroma side needs **no secret at all** anymore (see 9.1's `127.0.0.1` + SSH-tunnel-only
  design) — only the LiteLLM proxy key remains, because that's a genuine per-user credential by
  design (Section 10.4), not incidental friction.

Fill in your own values below (your SSH tunnel ports, model name, and the key issued to you in
Section 10.4), then run this cell once.

In [7]:
import re
from pathlib import Path
from IPython.display import Markdown, display


def show_py_sections(path, only=None):
    '''
    Show a real, committed .py file section by section, using its own "# ----" divider
    comments as headers. This cell only ever READS rag_workshop/*.py -- it never writes or
    overwrites it, so a fix you make directly in the file is never silently undone by
    re-running this cell (unlike the old %%writefile version this replaced).
    `only`: optional substring to filter which sections get displayed.
    '''
    lines = Path(path).read_text().splitlines()
    marker = re.compile(r"^#\s*-{10,}\s*$")
    marks = [i for i, l in enumerate(lines) if marker.match(l)]

    sections = []
    if marks and marks[0] > 0:
        sections.append(("Module setup (docstring, imports)", "\n".join(lines[:marks[0]])))
    for i in range(0, len(marks) - 1, 2):
        start, end = marks[i], marks[i + 1]
        title = lines[start + 1].lstrip("# ").strip()
        body_end = marks[i + 2] if i + 2 < len(marks) else len(lines)
        sections.append((title, "\n".join(lines[end + 1:body_end])))

    display(Markdown(f"**`{path}`** -- read live from disk below; this cell never writes to it."))
    for title, body in sections:
        if only and only.lower() not in title.lower():
            continue
        if not body.strip():
            continue
        display(Markdown(f"#### {title}\n```python\n{body.strip()}\n```"))


show_py_sections("rag_workshop/streamlit_app.py")


**`rag_workshop/streamlit_app.py`** -- read live from disk below; this cell never writes to it.

#### Module setup (docstring, imports)
```python
'''
Distributed hybrid RAG chat client.
Run with:  streamlit run rag_workshop/streamlit_app.py
Configuration lives in rag_workshop/.streamlit/secrets.toml (see Section 12.2) -- nothing
personal is hardcoded in this file, so it's safe to share/version this script itself.
'''
import logging
import pickle
import numpy as np
import streamlit as st
import chromadb
from FlagEmbedding import BGEM3FlagModel
import requests  # plain HTTP client -- talks only to OUR OWN LiteLLM proxy, no third-party account
from chunk_types import Chunk  # noqa: F401 -- needed to unpickle chunks.pkl (see notebook 1, 3.3)

# Streamlit's file watcher walks every loaded module (via transformers, pulled in by
# BGEM3FlagModel) to find source files to watch, which lazy-imports transformers' optional
# vision submodules -- several of those require torchvision (a normal dependency here, see
# pyproject.toml's "embeddings" extra). Belt-and-suspenders only: if an environment hasn't been
# resynced (`uv sync --extra embeddings`) and torchvision is genuinely missing, this just stops
# Streamlit from spamming a full traceback per submodule instead of the app failing to start.
logging.getLogger("streamlit.watcher.local_sources_watcher").setLevel(logging.ERROR)
```

#### Configuration -- loaded from .streamlit/secrets.toml, nothing hardcoded here
```python
REQUIRED_SECRETS = ["chroma_host", "chroma_port", "chroma_collection_name",
                     "litellm_proxy_url", "litellm_model_name", "litellm_key"]
missing = [k for k in REQUIRED_SECRETS if k not in st.secrets]
if missing:
    st.error(
        f"Missing secrets: {missing}. Fill in rag_workshop/.streamlit/secrets.toml "
        "(see notebook 2, Section 12.2) before running this app."
    )
    st.stop()

CHROMA_HOST = st.secrets["chroma_host"]
CHROMA_PORT = st.secrets["chroma_port"]
CHROMA_COLLECTION_NAME = st.secrets["chroma_collection_name"]

LITELLM_PROXY_URL = st.secrets["litellm_proxy_url"]
LITELLM_MODEL_NAME = st.secrets["litellm_model_name"]
MY_PARTICIPANT_KEY = st.secrets["litellm_key"]

LEXICAL_WEIGHTS_PATH = "rag_workshop/corpus/lexical_weights.pkl"
CHUNKS_PATH = "rag_workshop/corpus/chunks.pkl"
PAGE_TEXT_PATH = "rag_workshop/corpus/page_full_text_by_url.pkl"

# Small-to-big expansion (see notebook 1, Section 7.3): search with chunks, but let the
# top N results be substituted by their full page text when building the LLM's context.
EXPAND_TOP_N_TO_FULL_PAGE = 1
MAX_EXPANDED_CHARS = 3000

# How many previous (user, assistant) turns to resend to the LLM as conversation context, so
# follow-up questions ("and for GPU jobs?") work. Kept small on purpose: each turn adds tokens
# to every subsequent call, which costs against the participant's own max_budget (Section 10.4).
MAX_HISTORY_TURNS = 3
```

#### Confidence bands -- ports notebook 1, Section 7.1 to the distributed client. The more we
```python
RAG_ONLY_THRESHOLD = 0.80
RAG_SYNTHESIS_THRESHOLD = 0.60
RAG_HEDGE_THRESHOLD = 0.50

# A chunk pulled into the fused top-k isn't necessarily one the model actually leaned on --
# RRF can include a sparse-only match with no dense score at all (cosine_similarity=None), or a
# weak dense hit, purely on lexical overlap. Only link sources with an individually solid cosine,
# deduplicated by page (one chunk can't out-vote another chunk of the same page for the slot).
SOURCE_LINK_MIN_COSINE = 0.60
MAX_SOURCE_LINKS = 10

RAG_SYSTEM_PROMPT = (
    "You are a helpful technical assistant for a research computing center. "
    "Answer ONLY using the information in the provided context chunks. If the context does "
    "not contain the answer, say so explicitly instead of guessing. Always mention which "
    "source(s) (by title) you used. Keep answers concise and technically precise. Respond in "
    "the same language as the user's question. Earlier turns in the conversation may be "
    "included for context (e.g. a follow-up question) -- ground every factual claim in the "
    "context chunks provided with THIS question, not in what was said earlier."
)

LLM_ONLY_SYSTEM_PROMPT = (
    "You are a helpful technical assistant. No relevant passage was found in the lab's "
    "documentation for this question, so answer from your general knowledge instead. You MUST "
    "start your answer with an explicit note that this is general knowledge, not verified "
    "against the lab's own documentation, and that the user should double-check anything "
    "specific to their cluster/site. Respond in the same language as the user's question."
)

# Label + accent color per mode, used for the little badge shown above each answer.
MODE_STYLE = {
    "rag_only":      {"label": "Documentation — high confidence",   "color": "#16A34A"},
    "rag_synthesis": {"label": "Documentation + light synthesis",        "color": "#2563EB"},
    "rag_hedged":    {"label": "Documentation — low confidence",    "color": "#D97706"},
    "llm_only":      {"label": "General knowledge — not in docs",   "color": "#7C3AED"},
}


def sampling_for_similarity(top_similarity):
    '''Map a top cosine similarity to (mode_name, sampling_options) -- see notebook 1, 7.1.'''
    if top_similarity >= RAG_ONLY_THRESHOLD:
        return "rag_only", {"temperature": 0.1, "top_p": 0.7, "top_k": 10}
    elif top_similarity >= RAG_SYNTHESIS_THRESHOLD:
        return "rag_synthesis", {"temperature": 0.3, "top_p": 0.7, "top_k": 10}
    elif top_similarity >= RAG_HEDGE_THRESHOLD:
        return "rag_hedged", {"temperature": 0.7, "top_p": 0.7, "top_k": 10}
    else:
        return "llm_only", {"temperature": 0.7, "top_p": 0.9, "top_k": 40}
```

#### Cached resources -- loaded once per Streamlit session, not on every rerun
```python
@st.cache_resource
def load_embedding_model():
    return BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)

@st.cache_resource
def get_remote_collection():
    # No token, no Settings() -- Chroma is bound to 127.0.0.1 on Machine A (Section 9.1),
    # so the SSH tunnel itself is the only way in. Nothing to authenticate at this layer.
    client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    return client.get_collection(CHROMA_COLLECTION_NAME)

@st.cache_resource
def load_local_indexes():
    # lexical_weights.pkl is a plain list, one entry per chunk, in the same order as
    # chunks.pkl (see notebook 1, Section 4.3) -- it's not keyed by chunk_id. Rebuild the
    # chunk_id -> weights dict here exactly as notebook 1, Section 5.3 does in-memory.
    with open(LEXICAL_WEIGHTS_PATH, "rb") as f:
        sparse_weights = pickle.load(f)
    with open(CHUNKS_PATH, "rb") as f:
        chunks = pickle.load(f)
    with open(PAGE_TEXT_PATH, "rb") as f:
        page_full_text_by_url = pickle.load(f)
    sparse_index = {c.chunk_id: w for c, w in zip(chunks, sparse_weights)}
    chunk_lookup = {c.chunk_id: c for c in chunks}
    return sparse_index, chunk_lookup, page_full_text_by_url
```

#### Retrieval + generation (same logic as Part 1, targeting remote services)
```python
def dense_search_remote(query_dense_vec, collection, top_k=10):
    result = collection.query(query_embeddings=[query_dense_vec.tolist()], n_results=top_k)
    chunk_ids = result["ids"][0]
    distances = result["distances"][0]
    similarities = [1 - d for d in distances]
    return list(zip(chunk_ids, similarities))


def sparse_search_local(query_lexical_weights, sparse_index, bge_model, top_k=10):
    scores = []
    for chunk_id, chunk_weights in sparse_index.items():
        score = bge_model.compute_lexical_matching_score(query_lexical_weights, chunk_weights)
        scores.append((chunk_id, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


def reciprocal_rank_fusion(ranked_lists, k=60):
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for rank, (doc_id, _score) in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def hybrid_retrieve(query, bge_model, collection, sparse_index, chunk_lookup,
                     top_k_each=10, top_k_final=5):
    encoded = bge_model.encode([query], return_dense=True, return_sparse=True)
    query_dense = encoded["dense_vecs"][0]
    query_sparse = encoded["lexical_weights"][0]

    dense_results = dense_search_remote(query_dense, collection, top_k=top_k_each)
    sparse_results = sparse_search_local(query_sparse, sparse_index, bge_model, top_k=top_k_each)
    fused = reciprocal_rank_fusion([dense_results, sparse_results], k=60)[:top_k_final]

    dense_sim_lookup = dict(dense_results)
    enriched = []
    for chunk_id, rrf_score in fused:
        chunk = chunk_lookup[chunk_id]
        enriched.append({
            "text": chunk.text,
            "source_url": chunk.source_url,
            "source_title": chunk.source_title,
            "heading_path": getattr(chunk, "heading_path", ""),
            "rrf_score": rrf_score,
            "cosine_similarity": dense_sim_lookup.get(chunk_id),
        })
    return enriched


def build_context_block(results, page_full_text_by_url, expand_top_n=EXPAND_TOP_N_TO_FULL_PAGE):
    blocks = []
    for i, r in enumerate(results, 1):
        label = f"{r['source_title']} — {r['heading_path']}" if r.get("heading_path") else r["source_title"]
        if i <= expand_top_n and r["source_url"] in page_full_text_by_url:
            # Small-to-big: substitute the full page (truncated) instead of just the matched chunk.
            body = page_full_text_by_url[r["source_url"]][:MAX_EXPANDED_CHARS]
            blocks.append(f"[Source {i}: {label} ({r['source_url']}) -- FULL PAGE]\n{body}")
        else:
            blocks.append(f"[Source {i}: {label} ({r['source_url']})]\n{r['text']}")
    return "\n\n".join(blocks)


def call_remote_llm(messages, model=LITELLM_MODEL_NAME, sampling_options=None):
    '''
    Plain HTTP POST to OUR OWN LiteLLM proxy's OpenAI-compatible endpoint.
    No third-party SDK, no external account -- just requests + your workshop-issued key.
    `sampling_options` (temperature/top_p/top_k, see notebook 1, 7.1) are merged straight into
    the JSON body -- LiteLLM forwards non-OpenAI-standard fields like top_k through to Ollama.
    '''
    payload = {"model": model, "messages": messages}
    if sampling_options:
        payload.update(sampling_options)
    response = requests.post(
        f"{LITELLM_PROXY_URL}/v1/chat/completions",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {MY_PARTICIPANT_KEY}",
        },
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def generate_answer_remote(query, results, page_full_text_by_url, history=(), sampling_options=None):
    context = build_context_block(results, page_full_text_by_url)
    user_prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer the question using only the context above, and cite the source title(s) you used."
    )
    messages = [{"role": "system", "content": RAG_SYSTEM_PROMPT}]
    for role, content in history[-(2 * MAX_HISTORY_TURNS):]:
        messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": user_prompt})
    return call_remote_llm(messages, sampling_options=sampling_options)


def generate_answer_llm_only(query, history=(), sampling_options=None):
    messages = [{"role": "system", "content": LLM_ONLY_SYSTEM_PROMPT}]
    for role, content in history[-(2 * MAX_HISTORY_TURNS):]:
        messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": query})
    return call_remote_llm(messages, sampling_options=sampling_options)


def rag_chat(query, bge_model, collection, sparse_index, chunk_lookup, page_full_text_by_url, history=()):
    '''
    Full pipeline: retrieve, map the best cosine similarity to a confidence band (7.1), then
    generate accordingly -- grounded-and-near-extractive, grounded-with-synthesis, grounded-
    but-hedged, or (lowest band) general-knowledge-and-clearly-flagged. Returns
    (answer, results, top_similarity, mode) -- `mode` drives the badge/sources in the UI.
    '''
    results = hybrid_retrieve(query, bge_model, collection, sparse_index, chunk_lookup)
    cosine_scores = [r["cosine_similarity"] for r in results if r["cosine_similarity"] is not None]
    top_similarity = max(cosine_scores) if cosine_scores else 0.0

    mode, sampling_options = sampling_for_similarity(top_similarity)

    if mode == "llm_only":
        answer = generate_answer_llm_only(query, history, sampling_options)
        answer = (
            "\U0001F50E Nothing in the documentation clearly matched this question, so this "
            "answer comes from the model's general knowledge, **not** your lab's documentation:"
            "\n\n" + answer
        )
        return answer, results, top_similarity, mode

    answer = generate_answer_remote(query, results, page_full_text_by_url, history, sampling_options)
    if mode == "rag_hedged":
        answer = (
            "⚠️ I found some possibly related information, but I'm not fully "
            "confident it answers your exact question. Please double-check against the "
            "sources below.\n\n" + answer
        )
    return answer, results, top_similarity, mode


def confident_sources(results, min_cosine=SOURCE_LINK_MIN_COSINE, max_links=MAX_SOURCE_LINKS):
    '''
    Which sources are worth citing as clickable links: individually high cosine similarity,
    not just "present in the fused top-k" (see the module-level comment above). Deduplicated by
    (title, url) -- a page can contribute several chunks, each keeps only its best cosine for
    ranking -- and capped at `max_links`.
    '''
    best_cosine_by_source = {}
    for r in results:
        cos = r["cosine_similarity"]
        if cos is None or cos <= min_cosine:
            continue
        key = (r["source_title"], r["source_url"])
        if key not in best_cosine_by_source or cos > best_cosine_by_source[key]:
            best_cosine_by_source[key] = cos
    ranked = sorted(best_cosine_by_source.items(), key=lambda kv: kv[1], reverse=True)
    return [key for key, _cos in ranked[:max_links]]


def sources_markdown(results):
    sources = confident_sources(results)
    if not sources:
        return ""
    return "  \n".join(f"\U0001F517 [{title}]({url})" for title, url in sources)
```

#### Streamlit UI
```python
st.set_page_config(page_title="Lab RAG Assistant", page_icon="🧪", layout="centered")

st.markdown(
    """
    <style>
    .hero {
        background: linear-gradient(120deg, #0D9488 0%, #2563EB 100%);
        padding: 1.6rem 1.8rem;
        border-radius: 16px;
        color: white;
        margin-bottom: 1.2rem;
    }
    .hero h1 { margin: 0; font-size: 1.6rem; }
    .hero p { margin: 0.3rem 0 0 0; opacity: 0.92; font-size: 0.95rem; }
    .mode-badge {
        display: inline-block;
        padding: 2px 12px;
        border-radius: 999px;
        font-size: 0.78rem;
        font-weight: 600;
        margin-bottom: 0.5rem;
    }
    .sources-box {
        background: #F0FDFA;
        border-left: 3px solid #0D9488;
        padding: 0.6rem 0.9rem;
        border-radius: 8px;
        margin-top: 0.5rem;
        font-size: 0.9rem;
    }
    </style>
    <div class="hero">
        <h1>🧪 Lab RAG Assistant</h1>
        <p>Query encoding: local &nbsp;·&nbsp; Vector store: remote (Machine A) &nbsp;·&nbsp; LLM: remote (Machine B)</p>
    </div>
    """,
    unsafe_allow_html=True,
)


def mode_badge(mode):
    style = MODE_STYLE.get(mode)
    if not style:
        return
    st.markdown(
        f'<span class="mode-badge" style="background:{style["color"]}22; color:{style["color"]};">'
        f'{style["label"]}</span>',
        unsafe_allow_html=True,
    )


with st.sidebar:
    st.markdown("### 🔌 Connection status")
    try:
        bge_model = load_embedding_model()
        st.success("BGE-M3 loaded locally")
    except Exception as e:
        st.error(f"BGE-M3 failed to load: {e}")
        st.stop()

    try:
        collection = get_remote_collection()
        st.success(f"Connected to remote vector store ({collection.count()} chunks)")
    except Exception as e:
        st.error(f"Cannot reach remote vector store: {e}")
        st.stop()

    try:
        sparse_index, chunk_lookup, page_full_text_by_url = load_local_indexes()
        st.success(f"Loaded local sparse index ({len(sparse_index)} entries, {len(page_full_text_by_url)} pages)")
    except Exception as e:
        st.error(f"Cannot load local sparse index: {e}")
        st.stop()

    st.info(f"LLM proxy configured at {LITELLM_PROXY_URL} — first query will confirm connectivity.")

    st.divider()
    st.markdown("### 🎯 Confidence legend")
    for style in MODE_STYLE.values():
        st.markdown(
            f'<span class="mode-badge" style="background:{style["color"]}22; color:{style["color"]};">'
            f'{style["label"]}</span>',
            unsafe_allow_html=True,
        )

if "history" not in st.session_state:
    st.session_state.history = []  # list of dicts: role, content, and (for assistant) mode/sources/similarity/results

for entry in st.session_state.history:
    avatar = "🧑‍💻" if entry["role"] == "user" else "🧪"
    with st.chat_message(entry["role"], avatar=avatar):
        if entry["role"] == "assistant":
            mode_badge(entry.get("mode"))
        st.markdown(entry["content"])
        if entry.get("sources"):
            st.markdown(f'<div class="sources-box">{entry["sources"]}</div>', unsafe_allow_html=True)
        if entry.get("top_similarity") is not None:
            st.caption(f"Top cosine similarity: {entry['top_similarity']:.3f}")
        if entry.get("results") and entry.get("mode") != "llm_only":
            with st.expander("🔍 Retrieved chunks (debug)"):
                for r in entry["results"]:
                    sim = f"{r['cosine_similarity']:.3f}" if r["cosine_similarity"] is not None else "n/a"
                    st.markdown(f"**[{r['source_title']}]({r['source_url']})** — cos={sim}, rrf={r['rrf_score']:.4f}")
                    st.caption(r["text"][:300])

user_query = st.chat_input("Ask something about the lab documentation...")

if user_query:
    st.session_state.history.append({"role": "user", "content": user_query})
    with st.chat_message("user", avatar="🧑‍💻"):
        st.markdown(user_query)

    # Plain (role, content) turns only, for the LLM's own conversation memory (7.1 doesn't need
    # our badge/sources metadata, just what was actually said).
    llm_history = [(h["role"], h["content"]) for h in st.session_state.history[:-1]]

    with st.chat_message("assistant", avatar="🧪"):
        with st.spinner("Retrieving + generating..."):
            answer, results, top_similarity, mode = rag_chat(
                user_query, bge_model, collection, sparse_index, chunk_lookup, page_full_text_by_url,
                history=llm_history,
            )
        mode_badge(mode)
        st.markdown(answer)
        sources = sources_markdown(results) if mode != "llm_only" else ""
        if sources:
            st.markdown(f'<div class="sources-box">{sources}</div>', unsafe_allow_html=True)
        st.caption(f"Top cosine similarity: {top_similarity:.3f}")
        if mode != "llm_only":
            with st.expander("🔍 Retrieved chunks (debug)"):
                for r in results:
                    sim = f"{r['cosine_similarity']:.3f}" if r["cosine_similarity"] is not None else "n/a"
                    st.markdown(f"**[{r['source_title']}]({r['source_url']})** — cos={sim}, rrf={r['rrf_score']:.4f}")
                    st.caption(r["text"][:300])

    st.session_state.history.append({
        "role": "assistant",
        "content": answer,
        "mode": mode,
        "sources": sources,
        "top_similarity": top_similarity,
        "results": results,
    })
```

<a id="sec-12-4"></a>
### 12.4 💻 Launch the Streamlit app

In a terminal (not in this notebook — Streamlit needs its own process), from the directory
containing `rag_workshop/`:

```bash
streamlit run rag_workshop/streamlit_app.py
```

This opens a browser tab at `http://localhost:8501`. Make sure:

- Both SSH tunnels from Section 8 are still active,
- You filled in `rag_workshop/.streamlit/secrets.toml` with your real tunnel ports and the key
  issued to you in Section 10.4 (Section 12.2) — the app will show a clear error and stop if
  anything's missing, rather than failing cryptically,
- `lexical_weights.pkl`, `chunks.pkl`, and `page_full_text_by_url.pkl` from Part 1 are present
  under `rag_workshop/corpus/` (copy them from your Part 1 working directory if you're running
  this in a fresh folder).

<p align="center">
  <img src="assets/screenshots/streamlit_app_chat.png" alt="streamlit_app.py answering a question with no matching documentation" width="760"><br>
  <sub><i>streamlit_app.py — a question outside the corpus falls into the lowest confidence band
  (Section 7.1): the badge reads "General knowledge — not in docs" and the answer is explicitly
  flagged as not coming from the lab's documentation.</i></sub>
</p>

<p align="center">
  <img src="assets/screenshots/streamlit_app_retrieved_chunks.png" alt="streamlit_app.py's retrieved chunks debug expander" width="760"><br>
  <sub><i>The "Retrieved chunks (debug)" expander under an answer — each chunk's source page,
  cosine similarity, and RRF score (Section 6), exactly what fed the hybrid retrieval fusion for
  that query.</i></sub>
</p>

<a id="sec-13"></a>
## 13. End-to-end checklist

Before moving on to the Thunderbird plugin this afternoon, confirm all of the following:

- [ ] SSH tunnel to Machine A (vector store) is running.
- [ ] SSH tunnel to Machine B (LLM proxy) is running.
- [ ] `chroma run` is running on Machine A, and `remote_collection.count()` returns the expected
  number of chunks.
- [ ] Ollama + LiteLLM proxy are running on Machine B, and a test `chat.completions.create()`
  call succeeds with your personal API key.
- [ ] The Streamlit app loads without errors and all three sidebar checks (embedding model,
  vector store, sparse index) are green.
- [ ] Asking a question you know is covered by the corpus returns a confident, cited answer.
- [ ] Asking an off-topic question triggers the low-confidence / refusal branch.
- [ ] Stopping one SSH tunnel (simulate a network failure) produces a clear error in the sidebar,
  not a silent hang — if it doesn't, this is a good discussion point on error handling in
  distributed systems.


**What "it works" actually looks like, end to end** -- every box below is a section you already
built (8-12):

```mermaid
sequenceDiagram
    participant U as You (browser)
    participant S as Streamlit (your laptop)
    participant C as ChromaDB (Machine A, :8000 tunnel)
    participant L as LiteLLM (Machine B, :4000 tunnel)
    participant O as Ollama (Machine B)

    U->>S: Type a question
    S->>S: Encode query with BGE-M3 (dense + sparse, Section 9.3)
    S->>C: query(dense_vector) / query(sparse_vector)
    C-->>S: top-k chunks, dense list + sparse list
    S->>S: Reciprocal Rank Fusion -> ranked chunks + top similarity (nb.1 Section 6)
    S->>S: Pick confidence band from top similarity (nb.1 Section 7.1)
    S->>L: chat completion (prompt + context), your personal key
    L->>L: check key auth, budget, rate limit
    L->>O: forward to workshop-llm
    O-->>L: generated answer
    L-->>S: answer (usage logged to Postgres, Section 10.7)
    S-->>U: answer + clickable sources
```

If any step is missing or broken, the checklist above tells you which of Sections 8-12 to
revisit -- e.g. no answer at all usually means the tunnel or Chroma step, a `401`/`403` means the
LiteLLM key step, and an unsourced answer means the RRF/confidence step returned nothing above
threshold.


<a id="sec-14"></a>
## 14. Bonus: a secured, multi-user, themed client

### 14.1 What this section changes

Sections 8–13 gave every participant their **own** Streamlit process, on their **own** laptop,
reading their **own** `secrets.toml` with their **own** personal LiteLLM key. That's the right
setup for a one-day workshop, but it is not how a lab would run this for real: a lab typically
runs **one shared server** that everyone's browser connects to, which means three problems that
didn't exist before now need solving:

1. **Who is this browser tab, really?** With one shared process, you can no longer trust "whoever
   has the SSH tunnel open" — you need actual identity verification.
2. **Which LiteLLM key does this identified person get?** Not everyone should share one key (that
   defeats the whole point of Section 10's per-user budgets and revocation).
3. **Does person A ever see person B's chat history?** This one is, reassuringly, **already
   solved** by Streamlit itself — `st.session_state` is scoped per browser session on the server
   side, never shared between visitors, with or without any authentication layer. Section 12's
   app already benefits from this; this section just makes it explicit and ties it to a verified
   identity instead of an anonymous session.

We solve (1) and (2) below, and use two different mechanisms depending on your deployment:

| `auth_mode` | Mechanism | When to use it |
|---|---|---|
| `"oidc"` | Streamlit's native `st.login()` / `st.user`, via OpenID Connect | **Real lab deployment.** Delegates identity to your institution's own IdP (Entra ID, Shibboleth-via-OIDC, Okta...) — you verify nothing yourself, you just trust the IdP's signed token. |
| `"demo"` | [`streamlit-authenticator`](https://github.com/mkhorasani/Streamlit-Authenticator), a small username/password list | **This workshop only.** Registering a real OIDC client with an institutional IdP takes days of admin back-and-forth — not something to improvise live. This mode exists purely so you can see the *pattern* (auth gate → per-user key → isolated history) working end to end today. |

> ⚠️ Treat `"demo"` mode as a teaching aid, not a security boundary: a handful of shared
> username/password pairs, checked against plaintext-in-secrets.toml, is nowhere near what you'd
> want protecting real institutional data. For an actual deployment, use `"oidc"` and a real IdP.


<a id="sec-14-1"></a>
### 14.1 Per-user LiteLLM key vault

The missing piece from Section 10.4: a mapping from a **verified identity** (an email address,
typically) to that person's **own** LiteLLM key, looked up *server-side* — never sent by the
browser, never guessable from one user to the next.

For this notebook, that mapping lives in a `[user_keys]` table in `secrets.toml` (below), built
by the instructor/admin from the same `/key/generate` calls already shown in 10.4 — one real key
per identified participant. This is fine for a workshop of a few dozen people; for a real lab
deployment, replace this table with a proper secret manager (Vault, your cloud provider's KMS,
even just a database with restricted read access) — a plaintext TOML file is not where long-lived
institutional credentials belong.

The cell below rewrites `secrets.toml` from Section 12.2 as a **superset**: every key you already
filled in still works, plus the new `auth_mode`, `[user_keys]`, `[auth]`, and `[demo_auth]`
tables this section needs. If you already edited Section 12.2's file, copy your real values across
after running this cell.


**Who maps to what key, visually:**

```mermaid
flowchart LR
    subgraph Identity
        OIDC["OIDC login\n(institutional SSO)"]
        DEMO["Demo login\n(username + bcrypt password)"]
    end
    OIDC --> ID["verified identity\n(email or alias)"]
    DEMO --> ID
    ID --> VAULT["[user_keys] table\nin secrets.toml"]
    VAULT --> KEY["this user's own\nLiteLLM key"]
    KEY --> LL["LiteLLM proxy\n(Section 10)"]
```

The app never lets the browser choose or send its own key -- it looks the key up server-side from
the verified identity, the same way Section 10.4 issued one key per participant.


In [8]:
%%writefile rag_workshop/.streamlit/secrets.toml
# Personal / institutional configuration -- do NOT share this file or commit it to git.
# (Streamlit's default project .gitignore already excludes .streamlit/secrets.toml.)
# This is a superset of Section 12.2's file: fill in your real Chroma/LiteLLM values below,
# exactly as before, plus the new auth-related tables used by Section 14.

chroma_host = "localhost"      # SSH-tunneled Machine A, see Section 8-9
chroma_port = 8000
chroma_collection_name = "ccin2p3_docs"

litellm_proxy_url = "http://localhost:4000"   # SSH-tunneled Machine B, see Section 8-10
litellm_model_name = "workshop-llm"
litellm_key = "sk-REPLACE-WITH-YOUR-ISSUED-KEY"   # fallback key, used only if auth_mode = "demo"
                                                    # and the logged-in user has no entry below

# --- Which auth path this deployment uses (14.1) ---------------------------------------
# "oidc" -> real institutional SSO (production), needs [auth] filled in by your IdP admin.
# "demo" -> workshop-only username/password fallback, see 14.3. Do NOT use "demo" beyond today.
auth_mode = "demo"

# --- Per-user LiteLLM key vault (14.1) --------------------------------------------------
# One real participant key per identified user, from the /key/generate calls of Section 10.4.
[user_keys]
"alice@example.org" = "sk-REPLACE-WITH-ALICES-KEY"
"bob@example.org"   = "sk-REPLACE-WITH-BOBS-KEY"

# --- OIDC provider config (14.3) -- only read when auth_mode = "oidc" -------------------
[auth]
redirect_uri = "http://localhost:8501/oauth2callback"
cookie_secret = "REPLACE-WITH-A-LONG-RANDOM-STRING"
client_id = "REPLACE-WITH-YOUR-OIDC-CLIENT-ID"
client_secret = "REPLACE-WITH-YOUR-OIDC-CLIENT-SECRET"
server_metadata_url = "https://your-institution-idp.example.org/.well-known/openid-configuration"

# --- Demo-mode credentials (14.3) -- only read when auth_mode = "demo" ------------------
# streamlit-authenticator hashes plaintext passwords automatically at startup (auto_hash=True,
# the default) -- fine for a handful of workshop accounts. For many users, pre-hash instead with
# stauth.Hasher.hash_passwords(...) and pass auto_hash=False (see the library's own docs).
[demo_auth]
cookie_name = "labobots_demo_auth"
cookie_key = "REPLACE-WITH-A-LONG-RANDOM-STRING"
cookie_expiry_days = 1

[demo_auth.credentials.usernames.alice]
name = "Alice Dupont"
email = "alice@example.org"
password = "REPLACE-WITH-A-DEMO-PASSWORD"

[demo_auth.credentials.usernames.bob]
name = "Bob Martin"
email = "bob@example.org"
password = "REPLACE-WITH-A-DEMO-PASSWORD"


Overwriting rag_workshop/.streamlit/secrets.toml


<a id="sec-14-2"></a>
### 14.2 A theme, instead of Streamlit's defaults

Streamlit reads a `[theme]` table from `.streamlit/config.toml` (a sibling of `secrets.toml`,
safe to commit — it holds colors, not credentials) and applies it to every widget without any
per-widget styling work. Pick colors that fit your lab's branding; the values below are just a
reasonable, accessible starting point (clear contrast in both the sidebar and main area).


In [9]:
%%writefile rag_workshop/.streamlit/config.toml
# Safe to commit -- colors only, no secrets. See Section 14.2.
[server]
# Participants don't edit the app live, so auto-rerun-on-file-change isn't needed. This does
# NOT silence the noisy transformers/torchvision warnings below (Streamlit still scans loaded
# modules regardless of this setting) -- that's suppressed in the app itself, see its top.
fileWatcherType = "none"

[theme]
base = "light"
primaryColor = "#0D9488"            # accent: buttons, active menu item, links, chat input focus
backgroundColor = "#FFFFFF"         # main content area
secondaryBackgroundColor = "#F0FDFA"  # sidebar, code blocks, widget backgrounds (teal tint)
textColor = "#0F172A"
font = "sans serif"


Overwriting rag_workshop/.streamlit/config.toml


<a id="sec-14-3"></a>
### 14.3 The app: auth gate, per-user key, themed sidebar

`rag_workshop/streamlit_app_secure.py` is a real, already-committed file -- the cell below reads
and displays it (never writes it), spotlighting the one genuinely new part: an auth gate
(`require_login_oidc` / `require_login_demo`, reading `auth_mode` from `secrets.toml`), the
per-user LiteLLM key lookup from 14.1, and the themed sidebar from 14.2. Everything else in the
file is Section 12's app, unchanged.

**For a real workshop with more than a couple of test users, don't hand-fill `[demo_auth]` the
way the illustrative alice/bob entries in 14.1's `secrets.toml` do.** Since each participant runs
this app on their **own** laptop, their `secrets.toml` only ever needs to contain **their own**
username/password/key — never the whole group's. `rag_workshop/create_demo_accounts.py` generates
exactly that: one personalized, ready-to-use `secrets.toml` per participant, reusing the LiteLLM
key they were already issued (10.4) plus a freshly generated, bcrypt-hashed demo password (see the
README's "Participant login accounts" section for the full command and what it produces).

In [10]:
display(Markdown(
    "**`rag_workshop/streamlit_app_secure.py`** is Section 12's app (shown in full above) plus "
    "three additions: an auth gate, a per-user LiteLLM key lookup, and a themed sidebar (14.2). "
    "The config / confidence-bands / retrieval / generation sections are unchanged from 12.2's "
    "app, so only the new part is shown below -- open the file directly for the rest."
))
show_py_sections("rag_workshop/streamlit_app_secure.py", only="auth")


**`rag_workshop/streamlit_app_secure.py`** is Section 12's app (shown in full above) plus three additions: an auth gate, a per-user LiteLLM key lookup, and a themed sidebar (14.2). The config / confidence-bands / retrieval / generation sections are unchanged from 12.2's app, so only the new part is shown below -- open the file directly for the rest.

**`rag_workshop/streamlit_app_secure.py`** -- read live from disk below; this cell never writes to it.

#### 14.3.1 -- Auth gate: nothing below this block runs for an unverified visitor
```python
def require_login_oidc():
    """Production path: delegate identity entirely to the institution's OIDC provider."""
    if not st.user.is_logged_in:
        st.markdown(
            '<div class="hero"><h1>🔐 Lab RAG Assistant</h1>'
            '<p>Please sign in with your institutional account to continue.</p></div>',
            unsafe_allow_html=True,
        )
        st.button("Log in", on_click=st.login)
        st.stop()
    return st.user.get("name", st.user.email), st.user.email


def require_login_demo():
    """Workshop-only path: a small username/password list -- see the warning in 14.0."""
    import streamlit_authenticator as stauth

    demo_cfg = st.secrets["demo_auth"]
    credentials = {"usernames": {
        uname: dict(udata) for uname, udata in demo_cfg["credentials"]["usernames"].items()
    }}
    authenticator = stauth.Authenticate(
        credentials, demo_cfg["cookie_name"], demo_cfg["cookie_key"], demo_cfg["cookie_expiry_days"],
    )
    try:
        authenticator.login()
    except Exception as e:
        st.error(f"Login error: {e}")
        st.stop()

    status = st.session_state.get("authentication_status")
    if status is False:
        st.error("Incorrect username or password.")
        st.stop()
    if status is None:
        st.markdown(
            '<div class="hero"><h1>🔐 Lab RAG Assistant (workshop demo login)</h1>'
            '<p>Please enter your username and password.</p></div>',
            unsafe_allow_html=True,
        )
        st.stop()

    username = st.session_state["username"]
    name = st.session_state["name"]
    email = credentials["usernames"][username]["email"]
    st.session_state["_authenticator"] = authenticator  # kept for the logout button in the sidebar
    return name, email


if AUTH_MODE == "oidc":
    USER_NAME, USER_EMAIL = require_login_oidc()
elif AUTH_MODE == "demo":
    USER_NAME, USER_EMAIL = require_login_demo()
else:
    st.error(f"Unknown auth_mode {AUTH_MODE!r} in secrets.toml -- expected 'oidc' or 'demo'.")
    st.stop()


def get_user_litellm_key(email: str) -> str:
    """
    Per-user LiteLLM key lookup (14.1) -- replaces Section 12's single shared key. Falls back
    to secrets['litellm_key'] (if set) so the app still runs for an identified-but-unregistered
    visitor, clearly labelled as a fallback rather than failing outright.
    """
    user_keys = st.secrets.get("user_keys", {})
    if email in user_keys:
        return user_keys[email]
    fallback = st.secrets.get("litellm_key")
    if fallback:
        st.sidebar.warning(f"No personal LiteLLM key for {email} -- using the shared fallback key.")
        return fallback
    st.error(f"No LiteLLM key available for {email}. Ask an admin to register one (Section 10.4).")
    st.stop()


MY_LITELLM_KEY = get_user_litellm_key(USER_EMAIL)
```

<a id="sec-14-4"></a>
### 14.4 Launch, and what to check

```bash
streamlit run rag_workshop/streamlit_app_secure.py
```

In `"demo"` mode: if you're testing solo with 14.1's illustrative `secrets.toml`, log in as
`alice`/`bob` with the plaintext password you set there. If you generated per-participant files
with `rag_workshop/create_demo_accounts.py` (14.3), each participant logs in with their own alias
(e.g. `participant-07`) and the password from their row of the distribution list -- not
`alice`/`bob`. In `"oidc"` mode, you'll be redirected to your institution's login page instead.

Checklist, on top of Section 13's:

- [ ] Logging in as one user and asking a question, then logging out and back in as a *different*
  user (`alice`/`bob` if testing solo with 14.1's illustrative file, or two different
  `participant-XX` accounts if using 14.3's generated files) shows **two separate,
  empty-until-used** chat histories -- not one shared thread.
- [ ] The sidebar's "My account" page shows the correct name/email for whoever is logged in.
- [ ] If the logged-in user's email has no entry in `[user_keys]`, the app either falls back to
  `litellm_key` with a visible warning, or stops with a clear error -- never a silent failure.
  (Shouldn't happen with 14.3's generated files -- each one's `[user_keys]` entry always matches
  its own `[demo_auth]` login -- but worth knowing if you ever hand-edit one.)
- [ ] `LiteLLM`'s own `/key/info` endpoint (Section 10.6), checked per key, shows usage
  accumulating under the *individual* key that was actually used for each person's questions --
  confirming the per-user budget from Section 10.4 is doing real work here, not just existing on
  paper.
- [ ] The theme from Section 14.2 (colors, sidebar) is visibly different from Section 12's
  default Streamlit look.

This is still one deployment step short of production-ready (secret management, HTTPS in front
of Streamlit itself rather than just in front of Chroma/LiteLLM, log retention policy for who-
asked-what) -- but the *shape* of a secured, multi-user, on-brand lab assistant is now in place.

<p align="center">
  <img src="assets/screenshots/streamlit_app_secure_account.png" alt="streamlit_app_secure.py's My account page" width="620"><br>
  <sub><i>Right after logging in, the "My account" page (checklist above) shows name, email, and
  auth mode for whoever is logged in, plus a reminder that chat history and the LiteLLM key are
  both scoped to this one session (14.1).</i></sub>
</p>

<p align="center">
  <img src="assets/screenshots/streamlit_app_secure_themed.png" alt="streamlit_app_secure.py, signed in as participant-01, a low-confidence answer with no clickable source links, top cosine similarity 0.563" width="760"><br>
  <sub><i>streamlit_app_secure.py, signed in as <code>participant-01</code> (themed sidebar,
  14.2; scrolled past the top of this crop). This particular question only reached the
  "Documentation — low confidence" band (top cosine similarity 0.563, below the 0.60 threshold
  in <code>SOURCE_LINK_MIN_COSINE</code>) -- which is exactly why no source links are shown here:
  <code>confident_sources</code> only links sources above that threshold. A question that scores
  higher lands in "Documentation + light synthesis" instead, with clickable
  <code>sources_markdown</code> links, same mechanism as Section 12's base app.</i></sub>
</p>


<a id="whats-next"></a>
## What's next

This afternoon, we reuse **exactly this backend** — the remote vector store and the remote LLM
proxy, unchanged — and build a **Thunderbird plugin** that calls the same LiteLLM endpoint (and,
optionally, the same retrieval pipeline) to help you draft replies to your lab emails, directly
from your mail client instead of a browser tab.